In [21]:
import nltk
from nltk.corpus import movie_reviews
import pandas as pd
import random

# Download the movie reviews dataset
nltk.download('movie_reviews')

# Load the dataset
# Each file in `movie_reviews` contains text labeled as 'pos' (positive) or 'neg' (negative)
documents = [(list(movie_reviews.words(fileid)), category)
             for category in movie_reviews.categories()
             for fileid in movie_reviews.fileids(category)]

# Shuffle the data
random.shuffle(documents)

# Preview a sample
print(f"Sample document:\n{documents[0]}")


[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.


Sample document:
(['touchstone', 'pictures', 'and', 'spyglass', 'entertainment', 'presents', 'a', 'birnbaum', '/', 'barber', 'production', 'in', 'association', 'with', 'a', 'jackie', 'chan', 'films', 'limited', 'production', 'jackie', 'chan', 'owen', 'wilson', '"', 'shanghai', 'noon', '"', 'lucy', 'liu', 'music', 'by', 'randy', 'edelman', 'co', 'producers', 'ned', 'dowd', 'jules', 'daly', 'executive', 'producer', 'jackie', 'chan', 'willie', 'chan', 'and', 'solon', 'so', 'produced', 'by', 'roger', 'birnbaum', 'gary', 'barber', 'and', 'jonathan', 'glickman', 'written', 'by', 'alfred', 'gough', '&', 'miles', 'millar', 'directed', 'by', 'tom', 'dey', 'rated', 'pg', '-', '13', 'for', 'mild', 'language', ',', 'adult', 'situations', ',', 'drug', 'use', ',', 'martial', 'art', 'action', 'and', 'violence', '.', '107', 'minutes', '.', 'super', '35mm', '/', 'panavision', '(', '2', '.', '35', ':', '1', ')', 'what', 'can', 'you', 'say', 'about', 'jackie', 'chan', 'that', 'hasn', "'", 't', 'already',

In [22]:
# Convert the data into a DataFrame for easier handling
data = pd.DataFrame(documents, columns=['review', 'sentiment'])

# Combine words into a single string for each review
data['review'] = data['review'].apply(lambda x: ' '.join(x))

# Preview the DataFrame
print(data.head())


                                              review sentiment
0  touchstone pictures and spyglass entertainment...       pos
1  the premise of the new james wong film , final...       neg
2  dora ( fernanda montenegro ) sits behind a mak...       pos
3  voices . . . . . trey parker , matt stone , ge...       pos
4  i can already feel the hate letters pouring in...       pos


In [25]:
import nltk
nltk.download('punkt')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [26]:
data['review'] = data['review'].fillna('').astype(str)


In [29]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab') # Download the punkt_tab resource
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Define the preprocessing function
def preprocess_text(text):
    if not isinstance(text, str) or text.strip() == '':
        return ''
    # Remove special characters, numbers, and URLs
    text = re.sub(r'http\S+|www\S+|[^a-zA-Z\s]', '', text)
    # Tokenize and convert to lowercase
    tokens = word_tokenize(text.lower())
    # Remove stopwords
    tokens = [word for word in tokens if word not in stopwords.words('english')]
    return ' '.join(tokens)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [30]:
# Apply preprocessing safely
data['cleaned_review'] = data['review'].apply(preprocess_text)

# Display the preprocessed data
print(data[['review', 'cleaned_review']].head())


                                              review  \
0  plot : two teen couples go to a church party ,...   
1  the happy bastard ' s quick movie review damn ...   
2  it is movies like these that make a jaded movi...   
3  " quest for camelot " is warner bros . ' first...   
4  synopsis : a mentally unstable man undergoing ...   

                                      cleaned_review  
0  plot two teen couples go church party drink dr...  
1  happy bastard quick movie review damn yk bug g...  
2  movies like make jaded movie viewer thankful i...  
3  quest camelot warner bros first feature length...  
4  synopsis mentally unstable man undergoing psyc...  


In [31]:
from sklearn.model_selection import train_test_split

# Split data into training and testing sets
X = data['cleaned_review']
y = data['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")


Training samples: 1600, Testing samples: 400


In [32]:
from sklearn.feature_extraction.text import CountVectorizer

# Initialize CountVectorizer
vectorizer = CountVectorizer(max_features=5000)

# Fit and transform the training data
X_train_vec = vectorizer.fit_transform(X_train)

# Transform the testing data
X_test_vec = vectorizer.transform(X_test)

print(f"Vectorized data shape: {X_train_vec.shape}")


Vectorized data shape: (1600, 5000)


In [33]:
from sklearn.naive_bayes import MultinomialNB

# Initialize the classifier
model = MultinomialNB()

# Train the model
model.fit(X_train_vec, y_train)

# Predict on test data
y_pred = model.predict(X_test_vec)

print("Model training complete!")


Model training complete!


In [34]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# Display classification report
print("Classification Report:\n", classification_report(y_test, y_pred))

# Display confusion matrix
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.81
Classification Report:
               precision    recall  f1-score   support

         neg       0.79      0.84      0.81       199
         pos       0.83      0.78      0.81       201

    accuracy                           0.81       400
   macro avg       0.81      0.81      0.81       400
weighted avg       0.81      0.81      0.81       400

Confusion Matrix:
 [[167  32]
 [ 44 157]]


In [35]:
# Example custom reviews
custom_reviews = [
    "This movie was fantastic! The characters were well-developed and the story was engaging.",
    "I did not enjoy the film. It was too slow and the plot was predictable."
]

# Preprocess the custom reviews
custom_reviews_cleaned = [preprocess_text(review) for review in custom_reviews]

# Transform the reviews using the same vectorizer
custom_reviews_vec = vectorizer.transform(custom_reviews_cleaned)

# Predict sentiments
custom_predictions = model.predict(custom_reviews_vec)

# Display results
for review, sentiment in zip(custom_reviews, custom_predictions):
    print(f"Review: {review}\nPredicted Sentiment: {sentiment}\n")


Review: This movie was fantastic! The characters were well-developed and the story was engaging.
Predicted Sentiment: pos

Review: I did not enjoy the film. It was too slow and the plot was predictable.
Predicted Sentiment: neg

